# Sentiment Classification with Embeddings

**Module 8** — Embeddings in Practice

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

Let's build a complete sentiment classifier. The pipeline: tokenize → embed → aggregate → classify. We'll use pretrained embeddings and a simple architecture.


## Setup

Install required packages (skip if already installed):


In [ ]:
!pip install torch -q


## Code

Run each cell in order:


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SentimentClassifier(nn.Module):
    def __init__(self, pretrained_embeddings, num_classes=2, 
                 hidden_dim=128, freeze_embeddings=True):
        super().__init__()
        vocab_size, embed_dim = pretrained_embeddings.shape
        
        # Embedding layer (initialized with pretrained vectors)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings, dtype=torch.float32)
        )
        if freeze_embeddings:
            self.embedding.weight.requires_grad = False
        
        # Classifier head
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len) — word indices
        emb = self.embedding(x)           # (batch, seq_len, embed_dim)
        
        # Average pooling over the sequence
        avg = emb.mean(dim=1)             # (batch, embed_dim)
        
        # Classify
        h = F.relu(self.fc1(avg))         # (batch, hidden_dim)
        h = self.dropout(h)
        logits = self.fc2(h)              # (batch, num_classes)
        return logits

### Step 2: Training loop sketch


In [ ]:
def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for texts, labels in dataloader:
        optimizer.zero_grad()
        logits = model(texts)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(dataloader), correct / total

# Example usage:


## Key Takeaway

More sophisticated aggregation methods go beyond simple averaging:

**Max pooling**: Take the maximum value in each dimension across all words. Captures the most prominent features.

**Weighted averaging**: Weigh words by TF-IDF or attention. 'Wonderful' matters more than 'the'.

**Using an RNN/LSTM**: Process the sequence with an RNN, use the final hidden state. This captures word order — 'not bad' vs 'bad'.

*Complete sentiment classifier using pretrained embeddings*

---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
